# Reentrenar el modelo de riesgo (sin bajar los datos del INE)
Usa el panel ya procesado del **Release** (`resultados_para_colab.zip`) y los shapefiles del INE (también en el Release) para:
1. Reentrenar el modelo de riesgo (`src.risk_model`): Ridge, Lasso, Gradient Boosting y ensamble con pesos, variables de secciones vecinas, supuesto nacional, clasificador del 10% de mayor riesgo e intervalos de 80%.
2. Regenerar la cartografía ligera y el plan del agente.
3. Descargar un `resultados_para_colab.zip` nuevo para subirlo al Release.

*Entorno de ejecución → Ejecutar todas*. Tarda ~40–60 min (CPU normal, no hace falta GPU).

In [ ]:
%cd /content
import os
if not os.path.isdir('ParticipacionCiudadanaINE'):
    !git clone -q https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE.git
%cd /content/ParticipacionCiudadanaINE
!git pull -q
!pip -q install -r requirements.txt

### Datos procesados y cartografía (del Release)

In [ ]:
import os, glob, shutil, zipfile, urllib.request

URL = 'https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE/releases/latest/download/resultados_para_colab.zip'
urllib.request.urlretrieve(URL, 'resultados_para_colab.zip')
zipfile.ZipFile('resultados_para_colab.zip').extractall('tmp_zip')

# Acepta el zip tanto con los archivos en la raíz como dentro de data/processed/
encontrados = glob.glob('tmp_zip/**/riesgo_*.parquet', recursive=True)
assert encontrados, 'El zip no trae ningún riesgo_*.parquet'
os.makedirs('data/processed', exist_ok=True)
shutil.copytree(os.path.dirname(encontrados[0]), 'data/processed', dirs_exist_ok=True)
shutil.rmtree('tmp_zip'); os.remove('resultados_para_colab.zip')
print(sorted(os.listdir('data/processed')))

In [ ]:
%%bash
set -e
REL=https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE/releases/latest/download
mkdir -p data/geo && cd data/geo
for f in SECCION MUNICIPIO ENTIDAD; do
  curl -fsSL -o $f.zip "$REL/$f.zip" && mkdir -p $f && unzip -oq $f.zip -d $f && rm -f $f.zip
done
ls

### Modelo de riesgo (la parte larga)

In [ ]:
!python -W ignore -m src.risk_model

### Cartografía ligera y agente

In [ ]:
!python -W ignore -m src.geo
!python -m src.agent --anio 2027 --presupuesto-pct 10

### Resumen de métricas

In [ ]:
import json, pandas as pd
m = json.load(open('data/processed/riesgo_metricas.json', encoding='utf-8'))
cols = ['MAE_rel_pp', 'MAE_dentro_estado_pp', 'spearman', 'precision_top10', 'AUC_top10', 'MAE_pp', 'error_nacional_pp']
print('Modelo elegido:', m['modelo_elegido'], m.get('ensamble', {}))
print('Supuesto nacional:', m['supuesto_nacional']['regla_elegida'])
for b in ('validacion_promedio', 'prueba_2024'):
    print(f'
[{b}]'); display(pd.DataFrame(m[b]).T[cols])
print('
Top 10%:'); display(pd.DataFrame({(b, k): v for b, d in m['top10'].items() for k, v in d.items()}).T)
print('
Intervalos:'); display(pd.DataFrame({k: v for k, v in m['intervalos'].items() if isinstance(v, dict)}).T)

### Descargar resultados para el Release
Descarga `resultados_para_colab.zip` (incluye los mapas ya procesados) y súbelo al Release reemplazando el anterior (*Releases → v1.0 → Edit → quitar el viejo con la ✕ → arrastrar el nuevo*). Desde ese momento, `colab_app_rapida.ipynb` usa el modelo nuevo y levanta en pocos minutos.

In [ ]:
!rm -f /content/resultados_para_colab.zip && zip -qr /content/resultados_para_colab.zip data/processed && ls -lh /content/resultados_para_colab.zip
from google.colab import files
files.download('/content/resultados_para_colab.zip')

### (Opcional) Ver la app con el modelo nuevo

In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &
# Espera hasta 60 s a que aparezca la liga
for i in $(seq 1 30); do
  URL=$(grep -oE 'https://[a-z0-9-]+\.trycloudflare\.com' cloudflared.log | head -1)
  [ -n "$URL" ] && break; sleep 2
done
echo "${URL:-No apareció la liga; revisa cloudflared.log}"